# Cell markers, evidence ranks, durable CellGuide jobs and dataset export

All executed data are synthetic TSV/XLSX/HTTP JSON fixtures. The real public
source loaders, aggregation, per-cell completion receipts, GMT and Parquet
exports run offline. A ranking score has a **source-specific meaning**:
literature support count for curated sources, Marker Score for computational
CellGuide rows, and null for cross-species canonical CellGuide markers.


In [1]:
import json
import tempfile
from pathlib import Path
from unittest.mock import patch

import pandas as pd
import requests

work = Path(tempfile.mkdtemp(prefix="biodb-tutorial-"))

# A tiny HTTP boundary fixture: bioDB still performs its real parsing,
# validation, durable writes, and cache/receipt checks.


class Response:
    def __init__(self, data=None, body=None, status=200):
        self.data = data
        self.content = body if body is not None else json.dumps(data).encode()
        self.text = self.content.decode(errors="replace")
        self.status_code = status
        self.headers = {"Content-Length": str(len(self.content)), "ETag": '"fixture-v1"'}

    def json(self):
        return self.data

    def raise_for_status(self):
        if self.status_code >= 400:
            raise requests.HTTPError(f"synthetic HTTP {self.status_code}", response=self)

    def iter_content(self, chunk_size=65536):
        for start in range(0, len(self.content), chunk_size):
            yield self.content[start : start + chunk_size]

    def __enter__(self):
        return self

    def __exit__(self, *args):
        pass


class Session:
    def __init__(self, replies):
        self.replies, self.calls = iter(replies), []

    def get(self, url, **kwargs):
        self.calls.append(url)
        reply = next(self.replies)
        if isinstance(reply, Exception):
            raise reply
        return reply

In [2]:
from biodb import cellmarker, celltaxonomy, cellxgene, cellxgene_dataset, hf_datasets, markers

# One cache root contains the actual filenames each bulk reader expects.
ct_rows = []
cm_rows = []
for gene, pmids in [("SNAP25", ["1", "2"]), ("RBFOX3", ["3"])]:
    for pmid in pmids:
        ct_rows.append(
            {
                "Species": "Homo sapiens",
                "Tissue_standard": "brain",
                "Tissue_UberonOntology_ID": "UBERON_0000955",
                "Cell_standard": "neuron",
                "Specific_Cell_Ontology_ID": "CL_0000540",
                "Cell_Marker": gene,
                "Gene_ENTREZID": "25",
                "PMID": pmid,
            }
        )
        cm_rows.append(
            {
                "species": "Human",
                "tissue_type": "brain",
                "uberonongology_id": "UBERON_0000955",
                "cell_name": "neuron",
                "cellontology_id": "CL_0000540",
                "Symbol": gene,
                "GeneID": "25",
                "PMID": pmid,
            }
        )
pd.DataFrame(ct_rows).to_csv(work / celltaxonomy.RESOURCE_FILE, sep="\t", index=False)
pd.DataFrame(cm_rows).to_excel(work / cellmarker.FILES["all"], index=False)
ct = celltaxonomy.get_markers(cache_dir=work)
cm = cellmarker.query_markers("CL:0000540", cache_dir=work)
assert ct.set_index("gene_symbol").loc["SNAP25", "score"] == 2
assert cm.set_index("gene_symbol").loc["SNAP25", "rank"] == 1
assert set(ct["cell_ontology_id"]) == {"CL:0000540"}
print(ct[["species", "cell_ontology_id", "gene_symbol", "score", "rank"]])
print(cm[["species", "tissue_ontology_id", "gene_symbol", "score", "rank"]])

        species cell_ontology_id gene_symbol  score  rank
0  Homo sapiens       CL:0000540      RBFOX3      1     2
1  Homo sapiens       CL:0000540      SNAP25      2     1
  species tissue_ontology_id gene_symbol  score  rank
0   Human     UBERON:0000955      SNAP25      2     1
1   Human     UBERON:0000955      RBFOX3      1     2


Native CL/UBERON underscore IDs normalize to colon form. Species labels remain
as supplied by each source (`Human` versus `Homo sapiens`); filtering is exact,
case-insensitive membership, not a taxonomic synonym resolver. `map_to_cl=True`
can resolve missing cell labels through OLS in a live workflow.


In [3]:
from collections import Counter

# Mock the CDN HTTP boundary, including one first-attempt server failure.
# No marker, job, parser, receipt or dataset function is replaced.

calls = Counter()
failed_once = False


def cdn(url, **kwargs):
    global failed_once
    calls[url] += 1
    if url.endswith("latest_snapshot_identifier"):
        return Response(body=b"synthetic-snapshot")
    if url.endswith("celltype_metadata.json"):
        return Response({"CL:0000540": {"name": "neuron"}, "CL:0000127": {"name": "astrocyte"}})
    if url.endswith("tissue_metadata.json"):
        return Response({"UBERON:0000955": {"name": "brain"}})
    if "/computational_marker_genes/" in url:
        if url.endswith("CL_0000127.json") and not failed_once:
            failed_once = True
            return Response(status=503)
        return Response(
            [
                {
                    "groupby_dims": {
                        "organism_ontology_term_label": "Homo sapiens",
                        "tissue_ontology_term_label": "brain",
                    },
                    "symbol": "GENEA",
                    "gene_ontology_term_id": "ENSG1",
                    "marker_score": 0.9,
                    "specificity": 0.8,
                    "me": 1.0,
                    "pc": 0.7,
                }
            ]
        )
    if "/canonical_marker_genes/" in url:
        return Response(
            [{"symbol": "GENEB", "tissue": "brain", "publication": "synthetic citation"}]
        )
    raise AssertionError(f"Unexpected fixture URL: {url}")


with patch("requests.get", cdn):
    try:
        cellxgene.get_all_cellguide_markers(cache_dir=work, max_workers=1, progress=False)
    except RuntimeError as exc:
        assert "incomplete" in str(exc)
        print("One cell type failed; aggregate not published.")
    assert not list(work.glob("cellguide_markers*.parquet"))
    neuron_before = sum(n for url, n in calls.items() if "CL_0000540.json" in url)
    cg = cellxgene.get_all_cellguide_markers(cache_dir=work, max_workers=1, progress=False)
    neuron_after = sum(n for url, n in calls.items() if "CL_0000540.json" in url)
assert neuron_after == neuron_before  # committed cell type was not fetched again
assert len(list(work.glob("cellguide_jobs*/*.json"))) == 2
assert len(cg) == 4
print(
    cg[
        [
            "species",
            "tissue",
            "cell_ontology_id",
            "gene_symbol",
            "marker_type",
            "marker_score",
            "rank",
        ]
    ]
)

One cell type failed; aggregate not published.
        species tissue cell_ontology_id gene_symbol    marker_type  \
0  Homo sapiens  brain       CL:0000127       GENEA  computational   
1          None  brain       CL:0000127       GENEB      canonical   
2  Homo sapiens  brain       CL:0000540       GENEA  computational   
3           NaN  brain       CL:0000540       GENEB      canonical   

  marker_score  rank  
0          0.9     1  
1         None  None  
2          0.9   1.0  
3          NaN   NaN  


In [4]:
# The public multi-source aggregator preserves provenance and source metrics.
with patch("requests.get", cdn):
    combined = markers.load_markers(cache_dir=work)
    human = markers.load_markers(
        source=["celltaxonomy", "cellxgene"],
        species="Homo sapiens",
        cell_type="CL:0000540",
        tissue="UBERON:0000955",
        cache_dir=work,
    )
assert set(combined["source"]) == {"celltaxonomy", "cellmarker", "cellxgene"}
assert human["species"].eq("Homo sapiens").all()
# Canonical CellGuide rows have no species, so an explicit species filter drops them.
assert "GENEB" not in set(human["gene_symbol"])
print(combined[["source", "marker_type", "species", "gene_symbol", "score", "rank"]])

         source    marker_type       species gene_symbol  score  rank
0     cellxgene  computational  Homo sapiens       GENEA    0.9   1.0
1     cellxgene      canonical           NaN       GENEB    NaN  <NA>
2     cellxgene  computational  Homo sapiens       GENEA    0.9   1.0
3     cellxgene      canonical           NaN       GENEB    NaN  <NA>
4  celltaxonomy      canonical  Homo sapiens      RBFOX3    1.0   2.0
5  celltaxonomy      canonical  Homo sapiens      SNAP25    2.0   1.0
6    cellmarker      canonical         Human      RBFOX3    1.0   2.0
7    cellmarker      canonical         Human      SNAP25    2.0   1.0


In [5]:
from biodb.utils import read_gmt

ct_gmt = celltaxonomy.to_gmt(work / "markers.gmt", cache_dir=work)
print(read_gmt(ct_gmt, return_format="dict"))
# Local exports only: these APIs create real Parquet files and dataset cards.
ct_out = hf_datasets.build_celltaxonomy_dataset(work / "ct_dataset", cache_dir=work)
cm_out = hf_datasets.build_cellmarker_dataset(work / "cm_dataset", cache_dir=work)
with patch("requests.get", cdn):
    cg_out = cellxgene_dataset.build_dataset(
        work / "cg_dataset",
        cache_dir=work,
        organisms=["Homo sapiens"],
        include_deg=False,
        include_deg_rollup=False,
    )
for out in [ct_out, cm_out, cg_out]:
    paths = sorted(p.relative_to(out).as_posix() for p in out.rglob("*.parquet"))
    print(out.name, paths, "dataset card:", (out / "README.md").is_file())
assert "species" not in pd.read_parquet(ct_out / "markers" / "homo_sapiens.parquet").columns

{('CL:0000540', 'neuron'): ['SNAP25', 'RBFOX3']}
ct_dataset ['markers/homo_sapiens.parquet'] dataset card: True
cm_dataset ['markers/human.parquet'] dataset card: True
cg_dataset ['markers/canonical.parquet', 'markers/computational_homo_sapiens.parquet'] dataset card: True


## Live production recipes — not executed

The per-tissue WMG API and precomputed CellGuide CDN are separate interfaces.
Check cell/tissue availability first. `disease_vs_normal` makes a disease
contrast; a normal marker score is not a disease effect estimate.

```python
cellxgene.list_tissues(organism="Homo sapiens")
cellxgene.list_cell_types("brain")
cellxgene.query_markers("CL:0000540", tissue="brain", n_top=25)
cellxgene.list_diseases("lung")
cellxgene.disease_vs_normal("CL:0000084", tissue="lung", disease="MONDO:0004992")
cellxgene.cellguide_markers("CL:0000540", kind="both")
# Bulk jobs retain successful cell-type receipts on failure; rerun to resume.
cellxgene.get_all_cellguide_markers(cache_dir="cellguide", max_workers=4)

# Disease builds can involve many live queries. Opt in deliberately.
cellxgene_dataset.build_dataset("cellxgene-export", organisms=["Homo sapiens"],
    include_deg=True, include_deg_rollup=True, cache_dir="cellguide")
# MONDO rollups pool descendant disease groups versus normal.
# HF upload is a separate explicit publishing action, never executed here:
# hf_datasets.push_to_hub("ct-export", "your-namespace/celltaxonomy", private=True)
```

CellMarker uses the release-specific **CellMarker 2.0** download endpoint and
versioned default cache. It does not silently fall back to an unversioned mirror.
Optional Excel support is `biodb[celltype]`; publishing is `biodb[hf]`.
